# JobHop v2 Local LLM List Reranking Pilot

This Colab notebook implements the next LLM reranking pilot suggested by the supervisor.

Instead of asking the LLM to assign a 0-3 score to every candidate, this notebook gives the full retrieval top-100 list to a local instruction model and asks it to directly return the most relevant candidate IDs, ordered by relevance.

Protocol:

1. Load JobHop from Hugging Face: `aida-ugent/JobHop`.
2. Load ESCO metadata from Hugging Face: `danieldux/ESCO`.
3. Build last-step next-occupation validation cases.
4. Use BGE-M3 to retrieve top-100 candidate ESCO occupations.
5. Give the full top-100 candidate list to a local LLM.
6. Ask the LLM to return a JSON list of the most relevant candidate IDs.
7. Put the LLM-selected candidates first and append all remaining dense candidates in the original dense order.
8. Evaluate validation MRR / Recall@1 / Recall@5 / Recall@10.

This is a pilot notebook only. It does not modify the cluster scripts or the main experimental protocol.


In [ ]:
!pip -q install datasets pandas pyarrow requests tqdm sentence-transformers transformers accelerate bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 19.2 MB/s eta 0:00:00


## Configuration

This notebook uses only a local Hugging Face causal language model. No API key is needed.

The default LLM is `Qwen/Qwen2.5-7B-Instruct` loaded in 4-bit mode, which is stronger than the 3B pilot while still being more Colab-friendly. If your GPU has enough memory, set `LOAD_LLM_IN_4BIT = False`. If memory is tight, switch `LOCAL_LLM_MODEL` back to `Qwen/Qwen2.5-3B-Instruct`.


In [ ]:
import os

DATASET_ID = "aida-ugent/JobHop"
ESCO_DATASET_ID = "danieldux/ESCO"

MAX_EVAL_CASES = 50
RANDOM_SEED = 7

EMBEDDING_MODEL = "BAAI/bge-m3"
EMBED_BATCH_SIZE = 32
RETRIEVE_K = 100

# The LLM sees the full retrieval top-100 list and returns only the most relevant IDs.
LLM_INPUT_TOP_K = 100
LLM_RETURN_K_VALUES = [5, 10, 20, 50]

# Local LLM only. This can also be a local model path.
LOCAL_LLM_MODEL = os.environ.get("LOCAL_LLM_MODEL", "Qwen/Qwen2.5-7B-Instruct")
LOAD_LLM_IN_4BIT = True
LOCAL_LLM_MAX_NEW_TOKENS = 512

# Optional for China/HF mirror environments, e.g. os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
# os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

METRIC_KS = [1, 5, 10]

print({
    "dataset": DATASET_ID,
    "esco_metadata": ESCO_DATASET_ID,
    "max_eval_cases": MAX_EVAL_CASES,
    "embedding_model": EMBEDDING_MODEL,
    "retrieve_k": RETRIEVE_K,
    "llm_input_top_k": LLM_INPUT_TOP_K,
    "llm_return_k_values": LLM_RETURN_K_VALUES,
    "local_llm_model": LOCAL_LLM_MODEL,
    "load_llm_in_4bit": LOAD_LLM_IN_4BIT,
})


{'dataset': 'aida-ugent/JobHop', 'esco_metadata': 'danieldux/ESCO', 'max_eval_cases': 50, 'embedding_model': 'BAAI/bge-m3', 'retrieve_k': 100, 'llm_input_top_k': 100, 'llm_return_k_values': [5, 10, 20, 50], 'local_llm_model': 'Qwen/Qwen2.5-7B-Instruct', 'load_llm_in_4bit': True}


In [ ]:
import json
import random
import re
import time
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import requests
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "cuda", torch.version.cuda, "available", torch.cuda.is_available(), "device", device)
if device == "cuda":
    print(torch.cuda.get_device_name(0))

torch 2.11.0+cu128 cuda 12.8 available True device cuda
Tesla T4


## Data Loading and JobHop v2 Preprocessing

In [ ]:
SPLITS = ("train", "validation", "test")
REMOVE_UNKNOWN = True
COLLAPSE_CONSECUTIVE_DUPLICATES = True

def hf_tree(path=""):
    api = f"https://huggingface.co/api/datasets/{DATASET_ID}/tree/main"
    url = api if not path else f"{api}/{path}"
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    return response.json()

def list_parquet_files():
    files = []
    stack = [""]
    while stack:
        path = stack.pop()
        for item in hf_tree(path):
            item_path = item.get("path") or item.get("name")
            item_type = item.get("type")
            if not item_path:
                continue
            if item_type == "directory":
                stack.append(item_path)
            elif item_path.endswith(".parquet"):
                files.append(item_path)
    return sorted(files)

def split_from_path(path):
    lower = path.lower()
    for split in SPLITS:
        if re.search(rf"(^|/|_|-)({split})(/|_|-|\.|$)", lower):
            return split
    if "val" in lower:
        return "validation"
    return None

def hf_resolve_url(path):
    return f"https://huggingface.co/datasets/{DATASET_ID}/resolve/main/{path}"

def load_split(split, split_files):
    paths = split_files.get(split, [])
    if not paths:
        raise RuntimeError(f"No parquet files found for split={split}.")
    frames = [pd.read_parquet(hf_resolve_url(path)) for path in paths]
    df = pd.concat(frames, ignore_index=True)
    print(split, df.shape, paths)
    return df

def standardize_jobhop_schema(df):
    df = df.copy()
    rename = {}
    if "resume_id" in df.columns and "person_id" not in df.columns:
        rename["resume_id"] = "person_id"
    if "university_level" in df.columns and "university_studies" not in df.columns:
        rename["university_level"] = "university_studies"
    if rename:
        df = df.rename(columns=rename)
    required = ["person_id", "matched_code"]
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise RuntimeError(f"Missing required columns: {missing}; available columns={list(df.columns)}")
    return df

def normalize_code(value):
    if pd.isna(value):
        return None
    text = str(value).strip()
    if not text or text.lower() == "unknown":
        return None
    return text

def parse_year_quarter(value):
    if pd.isna(value):
        return (9999, 9)
    text = str(value)
    year_match = re.search(r"(19|20)\d{2}", text)
    year = int(year_match.group(0)) if year_match else 9999
    quarter_match = re.search(r"[Qq]([1-4])", text)
    quarter = int(quarter_match.group(1)) if quarter_match else 9
    return (year, quarter)

def make_trajectories(df):
    work = df.copy()
    work["matched_code"] = work["matched_code"].map(normalize_code)
    if REMOVE_UNKNOWN:
        work = work.dropna(subset=["matched_code"])
    work["_start_sort"] = work["start_date"].map(parse_year_quarter) if "start_date" in work.columns else [(9999, 9)] * len(work)
    work["_end_sort"] = work["end_date"].map(parse_year_quarter) if "end_date" in work.columns else [(9999, 9)] * len(work)
    sort_cols = ["person_id", "_start_sort", "_end_sort", "matched_code"]
    work = work.sort_values(sort_cols, kind="mergesort")
    trajectories = {}
    for person_id, group in work.groupby("person_id", sort=False):
        seq = []
        for _, row in group.iterrows():
            code = row["matched_code"]
            item = {
                "code": code,
                "university_studies": row.get("university_studies", None),
            }
            if COLLAPSE_CONSECUTIVE_DUPLICATES and seq and seq[-1]["code"] == code:
                continue
            seq.append(item)
        if len(seq) >= 2:
            trajectories[person_id] = seq
    return trajectories

def make_last_step_cases(trajectories):
    cases = []
    for person_id, seq in trajectories.items():
        cases.append({
            "person_id": person_id,
            "history": seq[:-1],
            "target": seq[-1]["code"],
            "university_studies": seq[-1].get("university_studies"),
        })
    return cases

In [ ]:
def normalize_esco_code(value):
    if pd.isna(value):
        return None
    return str(value).strip()

def load_esco_metadata():
    frames = []
    urls = [
        "https://huggingface.co/datasets/danieldux/ESCO/resolve/main/data/en/train.parquet",
        "https://huggingface.co/datasets/danieldux/ESCO/resolve/main/data/en/test.parquet",
    ]
    for url in urls:
        try:
            frames.append(pd.read_parquet(url))
        except Exception as exc:
            print("direct parquet load failed", url, exc)
    if not frames:
        ds = load_dataset("danieldux/ESCO", split="train")
        frames.append(ds.to_pandas())
    df = pd.concat(frames, ignore_index=True).drop_duplicates(subset=["ESCO_CODE"])
    lookup = {}
    for _, row in df.iterrows():
        code = normalize_esco_code(row.get("ESCO_CODE"))
        if code:
            lookup[code] = {
                "label": str(row.get("ESCO_OCCUPATION", "") or ""),
                "description": str(row.get("ESCO_DESCRIPTION", "") or ""),
            }
    return lookup

def occupation_label(metadata_lookup, code):
    meta = metadata_lookup.get(code, {})
    return meta.get("label") or f"ESCO code {code}"

def candidate_text(metadata_lookup, code):
    meta = metadata_lookup.get(code, {})
    label = meta.get("label") or f"ESCO code {code}"
    description = meta.get("description") or ""
    pieces = [f"ESCO occupation code: {code}.", f"Occupation title: {label}."]
    if description:
        pieces.append(f"Occupation description: {description}")
    return " ".join(pieces)

def query_text(metadata_lookup, case):
    history = "; ".join(occupation_label(metadata_lookup, item["code"]) for item in case["history"])
    codes = "; ".join(item["code"] for item in case["history"])
    edu = case.get("university_studies")
    edu_text = ""
    if edu is not None and not pd.isna(edu):
        edu_text = f" University studies: {edu}."
    return f"Career history occupations: {history}. Career history ESCO codes: {codes}.{edu_text} Predict the next ESCO occupation."

In [ ]:
parquet_files = list_parquet_files()
split_files = {split: [] for split in SPLITS}
for path in parquet_files:
    split = split_from_path(path)
    if split in split_files:
        split_files[split].append(path)

raw = {split: standardize_jobhop_schema(load_split(split, split_files)) for split in SPLITS}
metadata_lookup = load_esco_metadata()
trajectories = {split: make_trajectories(raw[split]) for split in SPLITS}

candidate_codes = sorted({item["code"] for seq in trajectories["train"].values() for item in seq})
candidate_set = set(candidate_codes)
validation_cases = make_last_step_cases(trajectories["validation"])
validation_cases = [case for case in validation_cases if case["target"] in candidate_set]

rng = random.Random(RANDOM_SEED)
validation_cases = rng.sample(validation_cases, min(MAX_EVAL_CASES, len(validation_cases)))

candidate_texts = [candidate_text(metadata_lookup, code) for code in candidate_codes]
validation_query_texts = [query_text(metadata_lookup, case) for case in validation_cases]

print("candidate occupations:", len(candidate_codes))
print("validation cases:", len(validation_cases))
print("example query:", validation_query_texts[0])
print("target:", validation_cases[0]["target"], occupation_label(metadata_lookup, validation_cases[0]["target"]))

train (1594827, 5) ['JobHop_v2_train.parquet']
validation (199587, 5) ['JobHop_v2_val.parquet']
test (198877, 5) ['JobHop_v2_test.parquet']
candidate occupations: 2982
validation cases: 50
example query: Career history occupations: sales processor; administrative assistant; insurance claims handler; supply chain assistant; forwarding manager. Career history ESCO codes: 5223.5; 3343.1; 3315.3; 3343.1.7; 3331.2. University studies: Secondary school. Predict the next ESCO occupation.
target: 3343.1.7 supply chain assistant


## Dense Retrieval Baseline

In [ ]:
model = SentenceTransformer(EMBEDDING_MODEL, device=device)

candidate_embeddings = model.encode(
    candidate_texts,
    batch_size=EMBED_BATCH_SIZE,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)
query_embeddings = model.encode(
    validation_query_texts,
    batch_size=EMBED_BATCH_SIZE,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

scores = query_embeddings @ candidate_embeddings.T
top_idx = np.argsort(-scores, axis=1)[:, :RETRIEVE_K]
dense_ranked = [[candidate_codes[j] for j in row] for row in top_idx]
dense_ranked_scores = [[float(scores[i, j]) for j in row] for i, row in enumerate(top_idx)]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.27GB            

... (9 outputs left out)

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Batches:   0%|          | 0/94 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
def evaluate_ranked_lists(cases, ranked_lists, ks=(1, 5, 10)):
    rows = []
    n = len(cases)
    ranks = []
    for case, ranked in zip(cases, ranked_lists):
        target = case["target"]
        try:
            rank = ranked.index(target) + 1
        except ValueError:
            rank = None
        ranks.append(rank)
    result = {"n": n, "MRR": float(np.mean([1.0 / r if r else 0.0 for r in ranks]))}
    for k in ks:
        result[f"Recall@{k}"] = float(np.mean([1.0 if r and r <= k else 0.0 for r in ranks]))
    return result, ranks

baseline_metrics, baseline_ranks = evaluate_ranked_lists(validation_cases, dense_ranked, METRIC_KS)
baseline_metrics

{'n': 50,
 'MRR': 0.0891419731050706,
 'Recall@1': 0.06,
 'Recall@5': 0.12,
 'Recall@10': 0.18}

## Local LLM List Reranking

For each validation query, the local LLM receives up to 100 dense retrieval candidates. Each candidate has a temporary numeric ID. The LLM must return only JSON:

```json
{"ranked_ids": [3, 17, 1, 42]}
```

Invalid IDs and duplicates are removed. Candidates not returned by the LLM are appended in their original dense retrieval order, so this is reranking rather than hard filtering.


In [ ]:
def compact_candidate_for_list(code, temporary_id):
    meta = metadata_lookup.get(code, {})
    label = meta.get("label") or f"ESCO code {code}"
    description = meta.get("description") or ""
    if len(description) > 120:
        description = description[:120] + "..."
    return f"[{temporary_id}] ESCO code: {code} | Title: {label} | Description: {description}"

def build_list_rerank_prompt(query, candidates, return_k):
    lines = [compact_candidate_for_list(code, idx + 1) for idx, code in enumerate(candidates)]
    candidate_block = "\n".join(lines)
    return f"""
You are reranking candidate ESCO occupations for next-occupation prediction.

Task:
Given a person's career history and a retrieval-ranked candidate list, select the {return_k} candidates that are most plausible as the person's next occupation.

Important rules:
- Use the temporary numeric IDs in square brackets, not the ESCO codes.
- Return the selected IDs ordered from most relevant to least relevant.
- Prefer occupations that are plausible next career steps from the history, not merely textually similar.
- Do not invent IDs.
- Do not include explanations.

Return exactly one valid JSON object in this format:
{{"ranked_ids": [1, 2, 3]}}

Career history query:
{query}

Candidate occupations:
{candidate_block}
""".strip()

def parse_json_object(text):
    text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if match:
            return json.loads(match.group(0))
        raise

def parse_ranked_ids(text, max_id):
    try:
        raw = parse_json_object(text)
        ids = raw.get("ranked_ids", [])
    except Exception:
        ids = re.findall(r"\d+", text)
    valid = []
    seen = set()
    invalid_count = 0
    duplicate_count = 0
    for item in ids:
        try:
            value = int(item)
        except Exception:
            invalid_count += 1
            continue
        if value < 1 or value > max_id:
            invalid_count += 1
            continue
        if value in seen:
            duplicate_count += 1
            continue
        seen.add(value)
        valid.append(value)
    return valid, invalid_count, duplicate_count

local_llm_tokenizer = None
local_llm_model = None

def load_local_llm():
    global local_llm_tokenizer, local_llm_model
    if local_llm_tokenizer is not None and local_llm_model is not None:
        return local_llm_tokenizer, local_llm_model
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    local_llm_tokenizer = AutoTokenizer.from_pretrained(LOCAL_LLM_MODEL, trust_remote_code=True)
    model_kwargs = {"trust_remote_code": True}
    if torch.cuda.is_available():
        model_kwargs["device_map"] = "auto"
        if LOAD_LLM_IN_4BIT:
            model_kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True,
            )
        else:
            model_kwargs["torch_dtype"] = torch.float16
    else:
        model_kwargs["torch_dtype"] = torch.float32
    local_llm_model = AutoModelForCausalLM.from_pretrained(LOCAL_LLM_MODEL, **model_kwargs)
    local_llm_model.eval()
    return local_llm_tokenizer, local_llm_model

def generate_local_llm(prompt):
    tokenizer, llm = load_local_llm()
    messages = [
        {"role": "system", "content": "You are a strict JSON-only reranker for occupation retrieval."},
        {"role": "user", "content": prompt},
    ]
    if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        text = "\n".join([f"{m['role'].upper()}: {m['content']}" for m in messages]) + "\nASSISTANT:"
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=30000).to(llm.device)
    with torch.no_grad():
        output_ids = llm.generate(
            **inputs,
            max_new_tokens=LOCAL_LLM_MAX_NEW_TOKENS,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated_ids = output_ids[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True)

def llm_select_candidate_ids(query, candidates, return_k):
    prompt = build_list_rerank_prompt(query, candidates, return_k)
    content = generate_local_llm(prompt)
    ranked_ids, invalid_count, duplicate_count = parse_ranked_ids(content, len(candidates))
    ranked_ids = ranked_ids[:return_k]
    selected_codes = [candidates[idx - 1] for idx in ranked_ids]
    return selected_codes, {
        "raw_output": content,
        "ranked_ids": ranked_ids,
        "invalid_id_count": invalid_count,
        "duplicate_id_count": duplicate_count,
        "num_returned_valid_ids": len(ranked_ids),
    }

def append_remaining_dense_order(selected_codes, dense_codes):
    selected_set = set(selected_codes)
    return selected_codes + [code for code in dense_codes if code not in selected_set]


In [ ]:
llm_outputs_by_return_k = {k: [] for k in LLM_RETURN_K_VALUES}
reranked_by_return_k = {k: [] for k in LLM_RETURN_K_VALUES}
failures = []

load_local_llm()
max_return_k = max(LLM_RETURN_K_VALUES)

for i, (query, ranked) in enumerate(tqdm(list(zip(validation_query_texts, dense_ranked)), desc="LLM list reranking")):
    candidates = ranked[:LLM_INPUT_TOP_K]
    try:
        selected_codes, info = llm_select_candidate_ids(query, candidates, max_return_k)
    except Exception as exc:
        failures.append({"case_index": i, "error": repr(exc)})
        selected_codes = []
        info = {
            "raw_output": "",
            "ranked_ids": [],
            "invalid_id_count": 0,
            "duplicate_id_count": 0,
            "num_returned_valid_ids": 0,
        }
    for return_k in LLM_RETURN_K_VALUES:
        selected_k = selected_codes[:return_k]
        llm_outputs_by_return_k[return_k].append({**info, "selected_codes": selected_k})
        reranked_by_return_k[return_k].append(append_remaining_dense_order(selected_k, ranked))

print("LLM failures:", len(failures))


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

... (2 outputs left out)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

LLM list reranking:   0%|          | 0/50 [00:00<?, ?it/s]

LLM failures: 0


## Compare Metrics


In [ ]:
summary_rows = [{"setting": "dense_baseline", "llm_return_k": 0, **baseline_metrics}]

gold_in_retrieval_top100 = float(np.mean([
    case["target"] in ranked[:LLM_INPUT_TOP_K]
    for case, ranked in zip(validation_cases, dense_ranked)
]))

for return_k in LLM_RETURN_K_VALUES:
    metrics, _ = evaluate_ranked_lists(validation_cases, reranked_by_return_k[return_k], METRIC_KS)
    outputs = llm_outputs_by_return_k[return_k]
    gold_selected = float(np.mean([
        case["target"] in output["selected_codes"]
        for case, output in zip(validation_cases, outputs)
    ]))
    gold_selected_ranks = []
    for case, output in zip(validation_cases, outputs):
        target = case["target"]
        if target in output["selected_codes"]:
            gold_selected_ranks.append(output["selected_codes"].index(target) + 1)
    summary_rows.append({
        "setting": "dense_plus_local_llm_list_rerank",
        "llm_return_k": return_k,
        **metrics,
        "gold_in_retrieval_top100": gold_in_retrieval_top100,
        "gold_selected_by_llm": gold_selected,
        "gold_selected_mean_rank": float(np.mean(gold_selected_ranks)) if gold_selected_ranks else None,
        "avg_returned_valid_ids": float(np.mean([output["num_returned_valid_ids"] for output in outputs])),
        "invalid_id_total": int(sum(output["invalid_id_count"] for output in outputs)),
        "duplicate_id_total": int(sum(output["duplicate_id_count"] for output in outputs)),
        "llm_failures": len(failures),
        "llm_failure_rate": len(failures) / max(1, len(validation_cases)),
    })

summary = pd.DataFrame(summary_rows)
display(summary)
best_idx = summary[summary["setting"] != "dense_baseline"]["MRR"].astype(float).idxmax()
best_return_k = int(summary.loc[best_idx, "llm_return_k"])
print("Best LLM_RETURN_K by validation MRR:", best_return_k)
print("LLM failures:", len(failures))


,setting,llm_return_k,n,MRR,Recall@1,Recall@5,Recall@10,gold_in_retrieval_top100,gold_selected_by_llm,gold_selected_mean_rank,avg_returned_valid_ids,invalid_id_total,duplicate_id_total,llm_failures,llm_failure_rate
0,dense_baseline,0,50,0.089142,0.06,0.12,0.18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,dense_plus_local_llm_list_rerank,5,50,0.081204,0.04,0.10,0.18,0.38,0.1,2.2,4.96,5.0,3.0,0.0,0.0
2,dense_plus_local_llm_list_rerank,10,50,0.081204,0.04,0.10,0.18,0.38,0.1,2.2,4.96,5.0,3.0,0.0,0.0
3,dense_plus_local_llm_list_rerank,20,50,0.081204,0.04,0.10,0.18,0.38,0.1,2.2,4.96,5.0,3.0,0.0,0.0
4,dense_plus_local_llm_list_rerank,50,50,0.081204,0.04,0.10,0.18,0.38,0.1,2.2,4.96,5.0,3.0,0.0,0.0


Best LLM_RETURN_K by validation MRR: 5
LLM failures: 0


## Inspect Examples and Save Outputs


In [ ]:
selected_reranked = reranked_by_return_k[best_return_k]
_, reranked_ranks = evaluate_ranked_lists(validation_cases, selected_reranked, METRIC_KS)
best_outputs = llm_outputs_by_return_k[best_return_k]

rows = []
for i, case in enumerate(validation_cases):
    target = case["target"]
    output = best_outputs[i]
    rows.append({
        "case_index": i,
        "person_id": case["person_id"],
        "target_code": target,
        "target_label": occupation_label(metadata_lookup, target),
        "baseline_rank": baseline_ranks[i],
        "reranked_rank": reranked_ranks[i],
        "gold_in_retrieval_top100": target in dense_ranked[i][:LLM_INPUT_TOP_K],
        "gold_selected_by_llm": target in output["selected_codes"],
        "query": validation_query_texts[i],
        "dense_top10": dense_ranked[i][:10],
        "llm_selected_codes": output["selected_codes"],
        "reranked_top10": selected_reranked[i][:10],
        "ranked_ids": output["ranked_ids"],
        "raw_output": output["raw_output"],
    })

details = pd.DataFrame(rows)
display(details.head(10))
summary.to_csv("jobhop_llm_list_reranking_pilot_summary.csv", index=False)
details.to_csv("jobhop_llm_list_reranking_pilot_details.csv", index=False)
with open("jobhop_llm_list_reranking_pilot_failures.json", "w", encoding="utf-8") as f:
    json.dump(failures, f, indent=2, ensure_ascii=False)
print("Saved jobhop_llm_list_reranking_pilot_summary.csv")
print("Saved jobhop_llm_list_reranking_pilot_details.csv")
print("Saved jobhop_llm_list_reranking_pilot_failures.json")


,case_index,person_id,target_code,target_label,baseline_rank,reranked_rank,gold_in_retrieval_top100,gold_selected_by_llm,query,dense_top10,llm_selected_codes,reranked_top10,ranked_ids,raw_output
0,0,148104,3343.1.7,supply chain assistant,3.0,2.0,True,True,Career history occupations: sales processor; a...,"[5223.5, 4311.2, 3343.1.7, 2423.1, 3343.1, 333...","[5223.5, 3343.1.7, 3331.2, 1324.8, 3322.1]","[5223.5, 3343.1.7, 3331.2, 1324.8, 3322.1, 431...","[1, 3, 10, 23, 26]","{""ranked_ids"": [1, 3, 10, 23, 26]}"
1,1,430074,3343.2,editorial assistant,NaN,NaN,False,False,Career history occupations: factory hand; inve...,"[9329.1, 4321.1, 2423.1, 1324.8.1, 4321.1.5, 2...","[1324.8, 1321.2.1.2, 7532.1, 1324.3.1, 1321.2.3]","[1324.8, 1321.2.1.2, 7532.1, 1324.3.1, 1321.2....","[9, 10, 13, 16, 17]","{""ranked_ids"": [9, 10, 13, 16, 17]}"
2,2,67623,5223.7.23,meat and meat products specialised seller,NaN,NaN,False,False,Career history occupations: cashier; babysitte...,"[5230.1, 5311.1.2, 5222.1.1, 2423.1, 5131.2, 2...","[5222.1.1, 5230.1, 5311.1.3, 5131.1, 5131.2]","[5222.1.1, 5230.1, 5311.1.3, 5131.1, 5131.2, 5...","[3, 1, 26, 27, 5]","{""ranked_ids"": [3, 1, 26, 27, 5]}"
3,3,179514,3422.4,sports instructor,NaN,NaN,False,False,Career history occupations: scopist; child car...,"[2635.3.2, 2423.1, 3412.4, 5311.1, 2342.1, 531...","[5311.1, 2342.1, 5312.1, 5311.1.2, 5311.1.3]","[5311.1, 2342.1, 5312.1, 5311.1.2, 5311.1.3, 2...","[4, 5, 6, 23, 9]","{""ranked_ids"": [4, 5, 6, 23, 9]}"
4,4,293935,4120.1,secretary,34.0,34.0,True,False,Career history occupations: administrative ass...,"[3343.1, 3343.1.2, 2359.1, 2423.1, 3343.3, 334...","[2359.1, 3343.3, 3343.1.2, 4416.1, 3313.1]","[2359.1, 3343.3, 3343.1.2, 4416.1, 3313.1, 334...","[3, 5, 2, 27, 25]","{""ranked_ids"": [3, 5, 2, 27, 25]}"
5,5,21485,5223.7.23,meat and meat products specialised seller,NaN,NaN,False,False,Career history occupations: cashier; accountin...,"[5230.1, 2423.1, 5222.1.1, 5223.4, 4311.2, 751...","[4311.2, 5222.1.1, 5223.6, 3313.1, 5223.3]","[4311.2, 5222.1.1, 5223.6, 3313.1, 5223.3, 523...","[5, 3, 12, 9, 10]","{""ranked_ids"": [5, 3, 12, 9, 10]}"
6,6,32238,3321.1,insurance broker,NaN,NaN,False,False,Career history occupations: sports instructor;...,"[2423.1, 2424.1, 3422.4, 2120.6, 5132.1, 5132....","[3422.4, 5132.1, 5223.6, 3412.3, 3423.2]","[3422.4, 5132.1, 5223.6, 3412.3, 3423.2, 2423....","[3, 5, 8, 9, 16]","{""ranked_ids"": [3, 5, 8, 9, 16]}"
7,7,372896,3122.4.12,paper mill supervisor,NaN,NaN,False,False,Career history occupations: market research an...,"[2431.4, 1221.3.2, 2431.11, 1324.3.1.6.3, 1221...","[1221.3.2, 2431.10, 4227.1, 2431.11, 1221.3.3]","[1221.3.2, 2431.10, 4227.1, 2431.11, 1221.3.3,...","[2, 17, 12, 3, 29]","{""ranked_ids"": [2, 17, 12, 3, 29]}"
8,8,242014,8131.7,fermenter operator,NaN,NaN,False,False,Career history occupations: kitchen assistant;...,"[2423.1, 5120.1, 9412.1, 8331.1, 8160.34, 3139...","[9412.1, 8331.1, 8160.34, 3139.1.1, 1341.1]","[9412.1, 8331.1, 8160.34, 3139.1.1, 1341.1, 24...","[3, 4, 5, 6, 20]","{""ranked_ids"": [3, 4, 5, 6, 20]}"
9,9,41706,9321.2,hand packer,NaN,NaN,False,False,Career history occupations: youth offending te...,"[2635.3.27, 2635.3.26, 2635.3.27.1, 2423.1, 93...","[2635.3.26, 2635.3.2, 2635.3, 2635.1, 2423.1]","[2635.3.26, 2635.3.2, 2635.3, 2635.1, 2423.1, ...","[2, 19, 12, 15, 4]","{""ranked_ids"": [2, 19, 12, 15, 4]}"


Saved jobhop_llm_list_reranking_pilot_summary.csv
Saved jobhop_llm_list_reranking_pilot_details.csv
Saved jobhop_llm_list_reranking_pilot_failures.json


## How to Interpret

Useful signs:

- MRR / Recall improves over the dense baseline.
- `gold_in_retrieval_top100` is high enough for the LLM to have a chance.
- `gold_selected_by_llm` is meaningfully above random.
- invalid / duplicate IDs and LLM failures are low.

Warning signs:

- The best LLM reranked result is worse than the dense baseline.
- The LLM frequently returns invalid IDs or too few IDs.
- `gold_in_retrieval_top100` is low, because then no reranker can recover many correct answers.

For a proper final experiment, choose the prompt and `LLM_RETURN_K` on validation only, then freeze them and evaluate test once.
